In [1]:
import numpy as np

### Model

In [2]:
x_hat = 0

p_x_given_y = np.array([[0.6, 0.4],
                        [0.9, 0.1]])

p_y_given_z = np.array([[0.2, 0.8],
                        [0.99, 0.01]])

p_z = np.array([0.4, 0.6])

### Message Passing

##### Mean-Field (VMP)

In [3]:
q_z = p_z.copy()
q_y = np.zeros(2)

for i in range(3):

    mu_y_right = p_x_given_y[:, x_hat]
    mu_y_left = np.exp(np.einsum('ij,i->j', np.log(p_y_given_z), q_z))
    q_y[...] = (mu_y_right * mu_y_left) / np.sum(mu_y_right * mu_y_left)

    mu_z_right = np.exp(np.einsum('ij,j->i', np.log(p_y_given_z), q_y))
    mu_z_left = p_z
    q_z[...] = (mu_z_right * mu_z_left) / np.sum(mu_z_right * mu_z_left)

    print("Iteration: " + str(i + 1))
    print("q_y: " + str(q_y))
    print("q_z: " + str(q_z))

Iteration: 1
q_y: [0.85779422 0.14220578]
q_z: [0.23971165 0.76028835]
Iteration: 2
q_y: [0.94024233 0.05975767]
q_z: [0.16145832 0.83854168]
Iteration: 3
q_y: [0.96172405 0.03827595]
q_z: [0.14480911 0.85519089]


##### Bethe (Sum-Product)

In [4]:
mu_y_right = p_x_given_y[:, x_hat]
mu_z_right = np.einsum('ij,j->i', p_y_given_z, mu_y_right)
mu_z_left = p_z
mu_y_left = np.einsum('ij,i->j', p_y_given_z, mu_z_left)
q_y = (mu_y_right * mu_y_left) / np.sum(mu_y_right * mu_y_left)
q_z = (mu_z_right * mu_z_left) / np.sum(mu_z_right * mu_z_left)
print("q_y: " + str(q_y))
print("q_z: " + str(q_z))

q_y: [0.57953568 0.42046432]
q_z: [0.48151333 0.51848667]


### Analytisch

In [5]:
p_y_and_z_and_xeq0 = np.einsum('i,ji,j->ji', p_x_given_y[:, x_hat], p_y_given_z, p_z)
p_xeq0 = np.einsum('ij->', p_y_and_z_and_xeq0)
p_y_and_z_given_xeq0 = p_y_and_z_and_xeq0 / p_xeq0
p_y_given_xeq0 = np.einsum('ij->j', p_y_and_z_given_xeq0)
print("p_y_given_xeq0: " + str(p_y_given_xeq0))

p_z_given_xeq0 = np.einsum('ij->i', p_y_and_z_given_xeq0)
print("p_z_given_xeq0: " + str(p_z_given_xeq0))

p_y_given_xeq0: [0.57953568 0.42046432]
p_z_given_xeq0: [0.48151333 0.51848667]
